## Структурная классификация белков

Хотя аминокислотная последовательность однозначно определяет трёхмерную структуру белка, предсказание полной 3D-структуры является сложной задачей.

В этом задании вам предстоит обучить несколько моделей для более простой задачи: предсказания типа белковой укладки по аминокислотной последовательности.


Датасет получен на основе базы данных [CATH](https://www.cathdb.info/wiki?id=data:index), представляющей иерархическую и стандартизированную классификацию белковых доменов по типам укладки. В датасете сделана подвыборка 9 архитектур укладки белков, для каждой собрано 2k примеров небольшой длины последовательности.

Данные и метки классов лежат в `assets/datasets/protein_fold`

In [1]:
from pathlib import Path

import pandas as pd
import torch
from torch import Tensor
from torch.utils.data import Dataset, DataLoader

import torch.optim as optim
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import math
import torch.nn.functional as F

Класс датасета уже написан:

In [2]:
class CathSequencesDataset(Dataset):
    def __init__(self, subset_csv: Path, labels_csv: Path, add_cls_token: bool = False) -> None:
        # словарь: 20 аминокислот + 2 спец символа:
        # _ - pad token, добивает последовательность до нужной длины
        # ? — токен класса, он будет нужен для обучения трансформера
        vocab = "_?ACDEFGHIKLMNPQRSTVWY"
        df = pd.read_csv(subset_csv)
        labels = pd.read_csv(labels_csv)["architecture"]
        self.add_cls_token = add_cls_token
        self.vocab = {char: i for i, char in enumerate(vocab)}
        self.label_dict = {name: i for i, name in enumerate(labels)}
        self.sequences = [self._encode_sequence(s) for s in df["sequence"]]
        self.labels = [self.label_dict[label] for label in df["architecture"]]

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.sequences[index], self.labels[index]

    def __len__(self) -> int:
        return len(self.sequences)
    
    def _encode_sequence(self, sequence: str) -> list[int]:
        tokens = [self.vocab[char] for char in sequence]
        if self.add_cls_token:
            tokens = [1] + tokens
        return tokens

    @staticmethod
    def collate_fn(batch: list[tuple[list[int], int]]) -> tuple[Tensor, Tensor]:
        encoded, labels = zip(*batch)
        max_len = max(map(len, encoded))
        x = torch.zeros((len(encoded), max_len), dtype=int)
        for i, seq in enumerate(encoded):
            x[i, : len(seq)] = torch.tensor(seq)

        return x, torch.tensor(list(labels))

    def batch_decode(self, out_tokens: Tensor) -> list[str]:
        # декодирование всего батча токенов
        # сделано неточно, так как при получении токена окончания генерации (!)
        # остаток последовательности нужно отбросить
        inv_vocab = {v: k for k, v in self.vocab.items()}
        decoded_strings: list[str] = []
        for x in out_tokens:
            decoded_strings.append("".join([inv_vocab[i] for i in x.tolist() if i > 2]))

        return decoded_strings

In [3]:
rootdir = Path("assets/datasets/protein_fold")
train_dataset = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)
tokens, label = train_dataset[0]
print("ID токенов:", tokens)
print("ID метки класса", label)

ID токенов: [14, 12, 19, 6, 11, 6, 14, 14, 10, 14, 10, 4, 18, 11, 12, 9, 17, 16, 18, 14, 5, 19, 18, 3, 19, 19, 19, 4, 19, 17, 4, 5, 4, 14, 5, 19, 10, 6, 13, 20, 21, 19, 4, 7, 19, 5, 19, 8, 13, 2, 10, 18, 10, 14, 16, 5, 5, 15, 21, 13, 2, 18, 21, 16, 19, 19, 17, 19, 11, 18, 19, 11, 8, 15, 4, 20, 11, 13, 7, 10, 5, 21, 10, 3, 10, 19, 17, 13, 10, 4, 11, 14, 2, 14, 9, 5, 10, 18, 9, 17]
ID метки класса 4


Последовательности в датасете имеют различную длину, поэтому для их объединения в один батч в `DataLoader(..., collate_fn=)` нужно будет передать фукнцию, которая правильно упаковывает список наблюдений в тензоры. Подходящая функция реализована в методе `CathSequencesDataset.collate_fn`

Посмотрим на мини-батч наблюдений:

In [4]:

train_dataset = CathSequencesDataset(subset_csv=rootdir / "train.csv", labels_csv=rootdir / "labels.csv", add_cls_token=False)
# sequences, labels = train_dataset.collate_fn([train_dataset[i] for i in range(4)])
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=train_dataset.collate_fn)
sequences, labels = next(iter(train_loader))
print("Последовательности:", sequences.shape)
print("Метки классов:", labels.shape)
sequences

Последовательности: torch.Size([4, 129])
Метки классов: torch.Size([4])


tensor([[14, 12, 19,  6, 11,  6, 14, 14, 10, 14, 10,  4, 18, 11, 12,  9, 17, 16,
         18, 14,  5, 19, 18,  3, 19, 19, 19,  4, 19, 17,  4,  5,  4, 14,  5, 19,
         10,  6, 13, 20, 21, 19,  4,  7, 19,  5, 19,  8, 13,  2, 10, 18, 10, 14,
         16,  5,  5, 15, 21, 13,  2, 18, 21, 16, 19, 19, 17, 19, 11, 18, 19, 11,
          8, 15,  4, 20, 11, 13,  7, 10,  5, 21, 10,  3, 10, 19, 17, 13, 10,  4,
         11, 14,  2, 14,  9,  5, 10, 18,  9, 17,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0],
        [ 7, 14, 11,  7, 17, 14,  5,  6, 18, 17, 12, 10, 14,  9, 13,  5,  4, 11,
          4,  5,  7,  9, 12, 19, 19, 21, 10, 16, 13,  9,  2,  7, 17,  7,  3, 15,
         11, 18,  6, 20,  5,  2, 17,  5, 16, 18,  9, 16, 17,  5,  2,  5,  4, 17,
         21,  8,  6, 17, 17,  2, 10, 12, 18,  2, 18,  6, 11, 17, 10, 10, 15,  5,
         19, 13, 12, 17,  4, 17,  2, 11,  4,  3, 19, 16,  4,  5,  2,  9, 13, 10,
      

### Задание 1 (5 баллов). Обучение классификатора на основе `nn.GRU`

Реализуйте классификатор последовательностей на основе GRU.
Ваш модуль будет иметь три нейросетевых блока:
1. Получение эмбеддингов для токенов
2. Получение скрытых состояний из рекуррентной сети
3. Классификатор последовательности на основе последнего скрытого состояния

В этом задании добейтесь точности классификации на валидационной выборке > 50%, для этого должно быть достаточно обучения в 15-20 эпох.

In [5]:
from torch import Tensor, nn


class GRUClassifier(nn.Module):
    """Модель для классификации"""

    def __init__(self, vocab_size: int, hidden_dim: int, n_classes: int) -> None:
        super().__init__()
        # эмбеддинги для аминокислот
        self.embed = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        # рекуррентный модуль
        self.rnn = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,  # обратите внимание на этот параметр! Это частый источник ошибок при работе с рекуррентными сетями
        )
        # голова модели
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, tokens: Tensor) -> Tensor:
        # B — размер батча
        # L — длина последовательности
        # H — размерность эмбеддингов
        # C — число классов
        # V — размер словаря токенов

        # получаем эмбеддинги токенов: B x L -> B x L x H
        x = self.embed(tokens)
        # обрабатываем последовательность токенов с помощью RNN: B x L x H -> B x L x H
        output, hidden = self.rnn(x)
        # извлекаем последние скрытые состояния: B x L x H -> B x H
        last_state = output[:, -1, :]

        # используем их для классификации последовательностей: B x H -> B x C
        logits = self.classifier(last_state)
        return logits


In [6]:
rootdir = Path("assets/datasets/protein_fold")
train_dataset = CathSequencesDataset(
    subset_csv=rootdir / "train.csv", 
    labels_csv=rootdir / "labels.csv", 
    add_cls_token=False
)
val_dataset = CathSequencesDataset(
    subset_csv=rootdir / "val.csv",
    labels_csv=rootdir / "labels.csv",
    add_cls_token=False
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_dataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_dataset.collate_fn)

In [7]:
vocab_size = len(train_dataset.vocab)
n_classes = len(train_dataset.label_dict)
hidden_dim = 128
model = GRUClassifier(vocab_size, hidden_dim, n_classes)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()

device = torch.device("cuda")
model.to(device)

GRUClassifier(
  (embed): Embedding(22, 128, padding_idx=0)
  (rnn): GRU(128, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=9, bias=True)
)

In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for tokens, labels in tqdm(loader, desc="Train"):
        tokens, labels = tokens.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(tokens)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for tokens, labels in tqdm(loader, desc="Val"):
            tokens, labels = tokens.to(device), labels.to(device)
            logits = model(tokens)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

In [9]:
n_epochs = 15
for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Acc={val_acc:.4f}")

print(f"Final Val Accuracy: {val_acc:.4f}")

Val: 100%|██████████| 32/32 [00:00<00:00, 536.83it/s]


Epoch 1: Train Loss=2.0335, Acc=0.2057 | Val Loss=2.0017, Acc=0.2470


Val: 100%|██████████| 32/32 [00:00<00:00, 495.48it/s]


Epoch 2: Train Loss=1.8593, Acc=0.2806 | Val Loss=1.7545, Acc=0.3430


Val: 100%|██████████| 32/32 [00:00<00:00, 613.83it/s]


...

Epoch 13: Train Loss=0.9227, Acc=0.6673 | Val Loss=1.1794, Acc=0.5930


Val: 100%|██████████| 32/32 [00:00<00:00, 636.90it/s]


Epoch 14: Train Loss=0.8600, Acc=0.6949 | Val Loss=1.1749, Acc=0.6100


Val: 100%|██████████| 32/32 [00:00<00:00, 628.80it/s]

Epoch 15: Train Loss=0.8193, Acc=0.7093 | Val Loss=1.2129, Acc=0.5960
Final Val Accuracy: 0.5960


### Задание 2 (5 баллов). Обучение классификатора на основе TransformerEncoder

Реализуйте классификатор последовательностей на основе блоков трансформера, приведённых в [notebooks/transformer_attention.ipynb](../notebooks/transformer_attention.ipynb).

Ваш модуль будет иметь три нейросетевых блока:
1. Получение начальных эмбеддингов для токенов и позиций
2. Преобразование эмбеддингов последовательностью блоков трансформера
3. Классификатор на основе финального эмбеддинга для токена <CLS>, который находится в начале каждой последовательности (не забудьте его включить в датасете!)

В этом задании добейтесь точности классификации на валидационной выборке > 60%, для этого должно быть достаточно обучения в 15-20 эпох.

In [10]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.W_q = nn.Linear(input_dim, hidden_dim)
        self.W_k = nn.Linear(input_dim, hidden_dim)
        self.W_v = nn.Linear(input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)
        _, _, D_k = k.shape

        scores = torch.bmm(q, k.permute(0, 2, 1)) / D_k**0.5
        attention_weights = scores.softmax(-1)
        return torch.bmm(attention_weights, v)

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [ScaledDotProductAttention(input_dim, hidden_dim) for i in range(n_heads)]
        )
        self.W_o = nn.Linear(n_heads * input_dim, input_dim)

    def forward(self, x: Tensor) -> Tensor:
        h = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.W_o(h)

In [12]:
class SelfAttentionLayer(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, n_heads: int) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(input_dim, hidden_dim, n_heads)
        self.norm1 = nn.LayerNorm(input_dim)
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(inplace=True),
            nn.Linear(input_dim, input_dim),
        )
        self.norm2 = nn.LayerNorm(input_dim)

    def forward(self, x: Tensor) -> Tensor:
        z = self.norm1(self.mha(x) + x)
        return self.norm2(self.mlp(z) + z)

In [13]:
class PositionalEncoding(nn.Module):
    pe: Tensor

    def __init__(self, encoding_size: int, L: int = 10000, max_len: int = 1000):
        super().__init__()
        # можно было бы считать налету, но мы заготовим буфер под любую допустимую длину
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, encoding_size, 2) * (-math.log(L) / encoding_size)
        )
        pe = torch.zeros(max_len, encoding_size)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, t: Tensor) -> Tensor:
        return self.pe[t]

In [14]:
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, pad_token_id: int = 0, max_position: int = 100):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=pad_token_id)
        self.position_embeddings = PositionalEncoding(hidden_size, max_len=max_position)
        # self.position_embeddings = nn.Embedding(max_position, hidden_size)

        self.register_buffer(
            "position_ids", torch.arange(max_position).expand((1, -1)), persistent=False
        )

    def forward(self, input_ids: Tensor) -> torch.Tensor:
        B, N = input_ids.size()
        position_ids = self.position_ids[:, : N]
        inputs_embeds = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        embeddings = inputs_embeds + position_embeddings
        return embeddings

In [15]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, n_layers: int, n_classes: int = 9, max_position: int = 300):
        super().__init__()
        # эмбеддинги токенов + позиций
        self.embeds = TransformerEmbeddings(vocab_size, hidden_size, pad_token_id=0, max_position=max_position)
        # последовательность SelfAttention модулей
        self.trunc = nn.ModuleList([
            SelfAttentionLayer(
                input_dim=hidden_size, 
                hidden_dim=hidden_size // 8,
                n_heads=8
            ) for _ in range(n_layers)
        ])
        # классификатор
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, x: Tensor) -> Tensor:
        x = self.embeds(x)
        for layer in self.trunc:
            x = layer(x)
        cls_emb = x[:, 0, :]
        logits = self.classifier(cls_emb)
        return logits

In [16]:
train_dataset = CathSequencesDataset(
    subset_csv=rootdir / "train.csv", 
    labels_csv=rootdir / "labels.csv", 
    add_cls_token=True
)
val_dataset = CathSequencesDataset(
    subset_csv=rootdir / "val.csv", 
    labels_csv=rootdir / "labels.csv", 
    add_cls_token=True
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=train_dataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=val_dataset.collate_fn)

In [17]:
hidden_size = 256
n_layers = 3
model = TransformerEncoder(vocab_size, hidden_size, n_layers, n_classes)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = CrossEntropyLoss()

model.to(device)

TransformerEncoder(
  (embeds): TransformerEmbeddings(
    (token_embeddings): Embedding(22, 256, padding_idx=0)
    (position_embeddings): PositionalEncoding()
  )
  (trunc): ModuleList(
    (0-2): 3 x SelfAttentionLayer(
      (mha): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x ScaledDotProductAttention(
            (W_q): Linear(in_features=256, out_features=32, bias=True)
            (W_k): Linear(in_features=256, out_features=32, bias=True)
            (W_v): Linear(in_features=256, out_features=256, bias=True)
          )
        )
        (W_o): Linear(in_features=2048, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classi

In [18]:
n_epochs = 10
for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Acc={val_acc:.4f}")

print(f"Final Val Accuracy: {val_acc:.4f}")

Val: 100%|██████████| 63/63 [00:00<00:00, 71.79it/s]


Epoch 1: Train Loss=1.4804, Acc=0.4554 | Val Loss=1.3352, Acc=0.4960


Val: 100%|██████████| 63/63 [00:00<00:00, 74.55it/s]


Epoch 2: Train Loss=1.3027, Acc=0.5244 | Val Loss=1.2273, Acc=0.5500


Val: 100%|██████████| 63/63 [00:00<00:00, 70.94it/s]


...

Epoch 8: Train Loss=0.8960, Acc=0.6849 | Val Loss=1.1103, Acc=0.6120


Val: 100%|██████████| 63/63 [00:00<00:00, 70.60it/s]


Epoch 9: Train Loss=0.8278, Acc=0.7083 | Val Loss=1.0860, Acc=0.6310


Val: 100%|██████████| 63/63 [00:00<00:00, 69.24it/s]

Epoch 10: Train Loss=0.7647, Acc=0.7296 | Val Loss=1.0855, Acc=0.6330
Final Val Accuracy: 0.6330


### Задание 3 (2 балла). Pre-norm vs Post-norm

Измените порядок следования механизма внимания, MLP и нормализаций на схему pre-normalized активаций (см. слайд **Постпроцессинг токенов: нормализация и MLP** в [slides/rnn_attention.pdf](../slides/rnn_attention.pdf)).

Запустите эксперимент с вашей лучшей конфигурацией модели из задания 2, заменив только определение `TransformerLayer`.

Изменилась ли точность или динамика обучения?